# 02 · `langchain-ollama`

`ChatOllama`로 같은 호출을 한다. 모델·옵션은 생성자 인자로 넣고, 메시지는 LangChain 메시지 객체를 쓴다. 앞으로 langchain·langgraph로 이어질 방식이다.

선행: Ollama가 켜져 있고 `qwen3:8b`가 받아져 있어야 한다(00-01).

In [1]:
import time
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

MODEL = "qwen3:8b"
OPTIONS = {"num_ctx": 4096, "temperature": 0.7}   # 세 노트북이 같은 값을 쓴다
THINK = False   # qwen3 사고 과정(<think>) 끄기 — 화면 처리는 Phase 3 전에 정한다
MESSAGES = [
    {"role": "system", "content": "너는 간결하게 한국어로 답하는 도우미다."},
    {"role": "user", "content": "스트리밍 응답이 왜 체감 속도를 높이는지 세 문장으로 설명해줘."},
]

llm = ChatOllama(
    model=MODEL,
    num_ctx=OPTIONS["num_ctx"],
    temperature=OPTIONS["temperature"],
    reasoning=THINK,   # False면 사고 과정을 끈다
)
LC_MESSAGES = [SystemMessage(MESSAGES[0]["content"]), HumanMessage(MESSAGES[1]["content"])]

## 1. 비스트리밍 — `invoke`
반환값은 `AIMessage`. 통계는 `response_metadata`에 들어 있다.

In [2]:
r = llm.invoke(LC_MESSAGES)
print(r.content)
print('eval_count =', r.response_metadata['eval_count'], '| eval_duration(ns) =', r.response_metadata['eval_duration'])

스트리밍 응답은 데이터를 작은 조각으로 나누어 즉시 전송하므로, 사용자가 대기 시간을 느끼지 않는다.  
데이터가 실시간으로 전달되기 때문에, 사용자는 빠르게 내용을 경험할 수 있다.  
이로 인해 전체 응답이 한 번에 로드되지 않고, 체감 속도가 빨라진다.
eval_count = 93 | eval_duration(ns) = 2084091000


## 2. 스트리밍 + tok/s — `stream`
`llm.stream()`은 `AIMessageChunk`를 하나씩 낸다. 통계는 `response_metadata`의 `done`이 `True`인 청크에 있다(공식 클라이언트의 마지막 청크와 같은 값). 그 뒤에 내용 없는 청크가 하나 더 올 수 있어서, "마지막 청크"가 아니라 `done`으로 찾는다.

In [3]:
t0 = time.perf_counter()
first = None
chunks = 0
meta = None
for chunk in llm.stream(LC_MESSAGES):
    if chunk.content:
        if first is None:
            first = time.perf_counter() - t0
        chunks += 1
        print(chunk.content, end="", flush=True)
    if chunk.response_metadata.get("done"):
        meta = chunk.response_metadata   # 통계가 실린 청크
print()
tps = meta["eval_count"] / (meta["eval_duration"] / 1e9)
print(f"청크 {chunks}개 | 첫 토큰까지 {first:.2f}s | 생성 {meta['eval_count']}토큰 | {tps:.1f} tok/s")

스트

리

밍

 응

답

...

아

집

니다

.


청크 69개 | 첫 토큰까지 0.05s | 생성 78토큰 | 44.7 tok/s


## 3. 멀티턴 — 메시지 객체 리스트
`SystemMessage`·`HumanMessage`·`AIMessage`를 순서대로 쌓아 보낸다. 역시 대화는 호출 쪽이 들고 있어야 한다.

In [4]:
history = list(LC_MESSAGES)
reply = "".join(c.content for c in llm.stream(history))
history.append(AIMessage(reply))
history.append(HumanMessage("방금 답을 한 문장으로 줄여줘."))

for chunk in llm.stream(history):
    print(chunk.content, end="", flush=True)
print()
print("메시지 수:", len(history))

스트

리

밍

 응

답

...

 만

듭

니다

.


메시지 수: 4
